## Install the required libraries

In [ ]:
# Install required libraries
!pip install openai PyPDF2 python-docx pandas
!pip install pytesseract Pillow pdf2image easyocr opencv-python

In [ ]:
# ========================================
# 1. IMPORTS AND SETUP
# ========================================

import openai
import json
import pandas as pd
from typing import Dict, List, Optional, Tuple
import PyPDF2
import docx
from datetime import datetime
import re
import cv2
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
import pytesseract
import easyocr
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configure pytesseract path (for Colab)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Test installation
print("Testing Tesseract installation...")
try:
    print(f"Tesseract version: {pytesseract.get_tesseract_version()}")
    print("✅ Tesseract installed successfully!")
except:
    print("❌ Tesseract installation failed!")

Testing Tesseract installation...
Tesseract version: 4.1.1
✅ Tesseract installed successfully!


In [ ]:
# Configure OpenAI API
openai.api_key = ""  # Replace with your actual API key

In [ ]:
models = openai.models.list()

for model in models.data:
    print(model.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-4o-audio-preview-2025-06-03
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-realtime-preview-2025-06-03
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
chatgpt-4o-latest
o1-preview-2024-09-12
o1-preview
o1-mini-2024-09-12
o1-mini
gpt-4o-realtime-preview-2024-10-01
gpt-4o-audio-preview-2024-10-01
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini

In [ ]:
# ========================================
# 2. CONTRACT DOCUMENT PROCESSOR
# ========================================

class ContractDocumentProcessor:
    """Handles document extraction and preprocessing"""

    def __init__(self):
        self.supported_formats = ['.pdf', '.txt', '.docx']

    def extract_text_from_pdf(self, file_path: str) -> str:
        """Extract text from PDF files"""
        try:
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() + "\n"
                return text
        except Exception as e:
            return f"Error extracting PDF: {str(e)}"

    def extract_text_from_docx(self, file_path: str) -> str:
        """Extract text from Word documents"""
        try:
            doc = docx.Document(file_path)
            text = ""
            for paragraph in doc.paragraphs:
                text += paragraph.text + "\n"
            return text
        except Exception as e:
            return f"Error extracting DOCX: {str(e)}"

    def extract_text_from_txt(self, file_path: str) -> str:
        """Extract text from TXT files"""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except Exception as e:
            return f"Error extracting TXT: {str(e)}"

    def process_document(self, file_path: str) -> str:
        """Process document based on file extension"""
        file_extension = file_path.lower().split('.')[-1]

        if file_extension == 'pdf':
            return self.extract_text_from_pdf(file_path)
        elif file_extension == 'docx':
            return self.extract_text_from_docx(file_path)
        elif file_extension == 'txt':
            return self.extract_text_from_txt(file_path)
        else:
            return f"Unsupported file format: {file_extension}"


In [ ]:
class ContractDocumentProcessor:
    """Enhanced processor with OCR capabilities for scanned documents"""

    def __init__(self):
        self.supported_formats = ['.pdf', '.txt', '.docx', '.jpg', '.jpeg', '.png']
        # Initialize both OCR engines
        self.easyocr_reader = easyocr.Reader(['en'])

        # Configure pytesseract
        pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

    def extract_text_from_pdf(self, file_path: str) -> str:
        """Extract text from PDF files"""
        try:
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() + "\n"
                return text
        except Exception as e:
            return f"Error extracting PDF: {str(e)}"

    def extract_text_from_docx(self, file_path: str) -> str:
        """Extract text from Word documents"""
        try:
            doc = docx.Document(file_path)
            text = ""
            for paragraph in doc.paragraphs:
                text += paragraph.text + "\n"
            return text
        except Exception as e:
            return f"Error extracting DOCX: {str(e)}"

    def extract_text_from_txt(self, file_path: str) -> str:
        """Extract text from TXT files"""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except Exception as e:
            return f"Error extracting TXT: {str(e)}"

    def is_scanned_pdf(self, file_path: str) -> bool:
        """Detect if PDF contains scanned images vs searchable text"""
        try:
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text = ""
                # Check first few pages
                pages_to_check = min(3, len(reader.pages))
                for i in range(pages_to_check):
                    text += reader.pages[i].extract_text()

                # If very little text extracted, likely scanned
                words = len(text.strip().split())
                print(f"Extracted {words} words from PDF. {'Scanned' if words < 20 else 'Searchable'} PDF detected.")
                return words < 20
        except:
            return True

    def preprocess_image_for_ocr(self, image):
        """Preprocess image for better OCR results"""
        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image

        # Remove noise
        denoised = cv2.fastNlMeansDenoising(gray)

        # Apply threshold to get better contrast
        _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        return thresh

    def extract_text_with_tesseract(self, image) -> str:
        """Extract text using Tesseract OCR"""
        try:
            # Configure Tesseract for better accuracy
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(image, config=custom_config)
            return text
        except Exception as e:
            print(f"Tesseract OCR failed: {str(e)}")
            return ""

    def extract_text_with_easyocr(self, image) -> str:
        """Extract text using EasyOCR"""
        try:
            results = self.easyocr_reader.readtext(image)
            text = ""
            for (bbox, detected_text, confidence) in results:
                if confidence > 0.3:  # Include text with reasonable confidence
                    text += detected_text + " "
            return text
        except Exception as e:
            print(f"EasyOCR failed: {str(e)}")
            return ""

    def extract_text_with_ocr(self, file_path: str) -> str:
        """Extract text from scanned PDFs using both OCR engines"""
        try:
            print("Converting PDF pages to images...")
            # Convert PDF pages to images with higher DPI for better OCR
            images = convert_from_path(file_path, dpi=300)
            extracted_text = ""

            for i, pil_image in enumerate(images):
                print(f"Processing page {i+1}/{len(images)} with OCR...")

                # Convert PIL image to OpenCV format
                cv_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

                # Preprocess image
                processed_image = self.preprocess_image_for_ocr(cv_image)

                # Try Tesseract first (usually better for documents)
                tesseract_text = self.extract_text_with_tesseract(Image.fromarray(processed_image))

                # If Tesseract doesn't give good results, try EasyOCR
                if len(tesseract_text.strip()) < 50:
                    print(f"  Tesseract gave limited results, trying EasyOCR...")
                    easyocr_text = self.extract_text_with_easyocr(processed_image)
                    page_text = easyocr_text if len(easyocr_text) > len(tesseract_text) else tesseract_text
                else:
                    page_text = tesseract_text

                extracted_text += f"\n=== Page {i+1} ===\n{page_text}\n"
                print(f"  Extracted {len(page_text)} characters from page {i+1}")

            print("OCR extraction completed.")
            return extracted_text

        except Exception as e:
            return f"OCR extraction failed: {str(e)}"

    def extract_text_from_image(self, file_path: str) -> str:
        """Extract text from image files (JPG, PNG, etc.)"""
        try:
            print("Processing image file with OCR...")

            # Load image
            image = cv2.imread(file_path)
            if image is None:
                return "Error: Could not load image file"

            # Preprocess image
            processed_image = self.preprocess_image_for_ocr(image)

            # Try Tesseract first
            tesseract_text = self.extract_text_with_tesseract(Image.fromarray(processed_image))

            # If Tesseract doesn't give good results, try EasyOCR
            if len(tesseract_text.strip()) < 20:
                print("  Tesseract gave limited results, trying EasyOCR...")
                easyocr_text = self.extract_text_with_easyocr(processed_image)
                final_text = easyocr_text if len(easyocr_text) > len(tesseract_text) else tesseract_text
            else:
                final_text = tesseract_text

            print("Image OCR extraction completed.")
            return final_text

        except Exception as e:
            return f"Image text extraction failed: {str(e)}"

    def process_document(self, file_path: str) -> str:
        """Enhanced document processing with automatic OCR detection"""
        file_extension = file_path.lower().split('.')[-1]

        print(f"Processing file: {file_path}")
        print(f"File extension: {file_extension}")

        if file_extension == 'pdf':
            # First check if PDF is scanned or searchable
            if self.is_scanned_pdf(file_path):
                print("🔍 Scanned PDF detected - Using OCR extraction...")
                return self.extract_text_with_ocr(file_path)
            else:
                print("📄 Searchable PDF detected - Using standard extraction...")
                return self.extract_text_from_pdf(file_path)

        elif file_extension in ['jpg', 'jpeg', 'png']:
            print("🖼️ Image file detected - Using OCR extraction...")
            return self.extract_text_from_image(file_path)

        elif file_extension == 'docx':
            print("📝 Word document detected - Using standard extraction...")
            return self.extract_text_from_docx(file_path)

        elif file_extension == 'txt':
            print("📋 Text file detected - Using standard extraction...")
            return self.extract_text_from_txt(file_path)
        else:
            return f"❌ Unsupported file format: {file_extension}"


In [ ]:
# ========================================
# 3. INDUSTRY-SPECIFIC KNOWLEDGE BASE
# ========================================

INDUSTRY_CONTEXTS = {
    "garment": {
        "critical_clauses": [
            "Quality Standards and Specifications",
            "Delivery Timelines and Penalties",
            "Material Sourcing Requirements",
            "Labor Compliance Standards",
            "Environmental Regulations",
            "Force Majeure for Supply Chain",
            "Payment Terms for Bulk Orders",
            "Inspection and Quality Control"
        ],
        "risk_factors": [
            "Supply chain disruption",
            "Quality control failures",
            "Labor law violations",
            "Environmental compliance",
            "Seasonal demand fluctuations"
        ]
    },
    "it": {
        "critical_clauses": [
            "Data Protection and Privacy",
            "Intellectual Property Rights",
            "Software Licensing Terms",
            "Service Level Agreements",
            "Liability Limitations",
            "Confidentiality and NDAs",
            "Maintenance and Support",
            "Termination and Data Return"
        ],
        "risk_factors": [
            "Data breaches and privacy violations",
            "IP infringement disputes",
            "Technology obsolescence",
            "Service interruptions",
            "Cybersecurity threats"
        ]
    },
    "construction": {
        "critical_clauses": [
            "Safety Regulations and Protocols",
            "Milestone Payment Schedules",
            "Material Quality Standards",
            "Weather and Delay Provisions",
            "Subcontractor Management",
            "Insurance and Bonding",
            "Change Order Procedures",
            "Final Inspection and Acceptance"
        ],
        "risk_factors": [
            "Construction delays and overruns",
            "Safety incidents and liability",
            "Material price fluctuations",
            "Weather-related delays",
            "Regulatory compliance issues"
        ]
    },

    "general": {
        "critical_clauses": [
            "Payment Terms and Conditions",
            "Governing Law and Jurisdiction",
            "Force Majeure Clause",
            "Termination Rights",
            "Confidentiality Agreement",
            "Dispute Resolution Mechanism",
            "Limitation of Liability",
            "Warranties and Representations"
        ],
        "risk_factors": [
            "Ambiguous contract language",
            "Missing or weak dispute clauses",
            "Jurisdictional confusion",
            "Late payments or defaults",
            "Unclear termination rights"
        ]
    }
}

In [ ]:
# ========================================
# 4. GPT-4 CONTRACT ANALYZER
# ========================================

class ContractAIAnalyzer:
    """Main AI analyzer using GPT-4 for contract intelligence"""

    def __init__(self, api_key: str):
        openai.api_key = api_key
        self.model = "gpt-4.1"
        self.max_tokens = 4000

    def create_analysis_prompt(self, contract_text: str, industry: str, language: str) -> str:
        """Create specialized prompt for contract analysis"""

        industry_context = INDUSTRY_CONTEXTS.get(industry.lower(), {})
        critical_clauses = industry_context.get("critical_clauses", [])
        risk_factors = industry_context.get("risk_factors", [])

        prompt = f"""
You are an expert ChainSight Contract AI Intelligence Analyst specializing in {industry.upper()} industry contracts.

DOCUMENT LANGUAGE: {language}
TARGET INDUSTRY: {industry.upper()}

CRITICAL CLAUSES FOR {industry.upper()} INDUSTRY:
{chr(10).join([f"- {clause}" for clause in critical_clauses])}

COMMON RISK FACTORS:
{chr(10).join([f"- {risk}" for risk in risk_factors])}

CONTRACT TEXT TO ANALYZE:
{contract_text}

ANALYSIS REQUIREMENTS:
1. Perform comprehensive risk assessment (scale 1-10, format like 5/10)
2. Identify missing critical clauses specific to {industry} industry
3. Detect potential legal, financial, and operational risks
4. Provide industry-specific recommendations
5. Suggest specific contract improvements

OUTPUT FORMAT (JSON):
{{
    "document_analysis": {{
        "industry": "{industry}",
        "language": "{language}",
        "analysis_date": "{datetime.now().strftime('%Y-%m-%d')}",
        "overall_risk_score": [1-10],
        "executive_summary": {{
            "critical_issues_count": [number],
            "missing_clauses_count": [number],
            "priority_level": "[High/Medium/Low]"
        }},
        "risk_assessment": [
            {{
                "category": "[Legal/Financial/Operational]",
                "severity": "[High/Medium/Low]",
                "description": "[detailed description]",
                "potential_impact": "[impact description]",
                "likelihood": "[High/Medium/Low]"
            }}
        ],
        "missing_critical_clauses": [
            {{
                "clause_name": "[clause name]",
                "importance": "[Critical/Important/Recommended]",
                "reason": "[why this clause is needed]",
                "suggested_text": "[sample clause text]"
            }}
        ],
        "identified_risks": [
            {{
                "risk_type": "[specific risk]",
                "severity": "[High/Medium/Low]",
                "current_protection": "[existing protection if any]",
                "mitigation_suggestion": "[how to address]"
            }}
        ],
        "improvement_recommendations": [
            {{
                "priority": [1-3],
                "category": "[Addition/Modification/Clarification]",
                "description": "[what needs to be changed]",
                "justification": "[why this change is needed]",
                "suggested_implementation": "[how to implement]"
            }}
        ],
        "compliance_check": {{
            "industry_standards": "[compliant/non-compliant/partial]",
            "regulatory_requirements": "[analysis of regulatory compliance]",
            "best_practices": "[adherence to industry best practices]"
        }}
    }}
}}

Provide thorough, industry-specific analysis with actionable recommendations.
"""
        return prompt

    def analyze_contract(self, contract_text: str, industry: str, language: str) -> Dict:
        """Analyze contract using GPT-4"""

        try:
            prompt = self.create_analysis_prompt(contract_text, industry, language)

            response = openai.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert Contract AI Intelligence Analyst with deep expertise in legal contract analysis, risk assessment, and industry-specific knowledge. Provide detailed, actionable analysis in the specified JSON format."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=self.max_tokens,
                temperature=0.3,  # Lower temperature for more consistent analysis
                response_format={"type": "json_object"}
            )

            analysis_result = json.loads(response.choices[0].message.content)
            return analysis_result

        except Exception as e:
            return {
                "error": f"Analysis failed: {str(e)}",
                "timestamp": datetime.now().isoformat()
            }

    def batch_analyze_contracts(self, contracts: List[Tuple[str, str, str, str]]) -> List[Dict]:
        """Analyze multiple contracts in batch

        Args:
            contracts: List of tuples (file_path, contract_text, industry, language)
        """
        results = []

        for file_path, contract_text, industry, language in contracts:
            print(f"Analyzing: {file_path}")
            analysis = self.analyze_contract(contract_text, industry, language)
            analysis['source_file'] = file_path
            results.append(analysis)

        return results

In [ ]:
# ========================================
# 5. TEST IMPLEMENTATION
# ========================================


# First upload a sample contract file to Colab
from google.colab import files
uploaded = files.upload()

# Get the uploaded filename
file_name = next(iter(uploaded))
# file_name = '/content/ennysports-contract-agreement.pdf'
print(f"Uploaded file: {file_name}")

Saving annex-beechwoodmsbkyssamplecontract.pdf to annex-beechwoodmsbkyssamplecontract.pdf
Uploaded file: annex-beechwoodmsbkyssamplecontract.pdf


In [ ]:

# Initialize the document processor
processor = ContractDocumentProcessor()

# Process the uploaded file
contract_text = processor.process_document(file_name)
print(f"Extracted text length: {len(contract_text)} characters")

Processing file: annex-beechwoodmsbkyssamplecontract.pdf
File extension: pdf
Extracted 1471 words from PDF. Searchable PDF detected.
📄 Searchable PDF detected - Using standard extraction...
Extracted text length: 9071 characters


In [ ]:
# Initialize the analyzer (replace with your actual API key)
analyzer = ContractAIAnalyzer(api_key=openai.api_key)  # Replace with your key

# Select industry and language for analysis
industry = "it"  # Can be "garment", "it", or "construction"
language = "English"  # Language of the contract

# Analyze the contract
analysis_result = analyzer.analyze_contract(contract_text, industry, language)

analysis_result



{'document_analysis': {'industry': 'it',
  'language': 'English',
  'analysis_date': '2025-07-21',
  'overall_risk_score': 8,
  'executive_summary': {'critical_issues_count': 7,
   'missing_clauses_count': 8,
   'priority_level': 'High'},
  'risk_assessment': [{'category': 'Legal',
    'severity': 'High',
    'description': 'Absence of Data Protection and Privacy clauses exposes both parties to significant regulatory and reputational risks, especially when handling sensitive beneficiary data.',
    'potential_impact': 'Potential violation of GDPR or other data privacy laws, leading to fines, sanctions, and loss of trust.',
    'likelihood': 'High'},
   {'category': 'Legal',
    'severity': 'High',
    'description': 'No Intellectual Property Rights (IPR) or Software Licensing Terms present, which is critical for IT contracts involving technology, software, or proprietary processes.',
    'potential_impact': 'Disputes over ownership or use of technology, software, or data; potential IP 

In [ ]:
# Print the JSON output
print("\nAnalysis Results:")
print(json.dumps(analysis_result, indent=2))


Analysis Results:
{
  "document_analysis": {
    "industry": "it",
    "language": "English",
    "analysis_date": "2025-07-21",
    "overall_risk_score": 8,
    "executive_summary": {
      "critical_issues_count": 7,
      "missing_clauses_count": 8,
      "priority_level": "High"
    },
    "risk_assessment": [
      {
        "category": "Legal",
        "severity": "High",
        "description": "Absence of Data Protection and Privacy clauses exposes both parties to significant regulatory and reputational risks, especially when handling sensitive beneficiary data.",
        "potential_impact": "Potential violation of GDPR or other data privacy laws, leading to fines, sanctions, and loss of trust.",
        "likelihood": "High"
      },
      {
        "category": "Legal",
        "severity": "High",
        "description": "No Intellectual Property Rights (IPR) or Software Licensing Terms present, which is critical for IT contracts involving technology, software, or proprietary pr